# 05 - Face Detection & Landmarks

This notebook demonstrates OCI Vision's face detection feature, which
locates faces in an image and identifies facial landmarks (eyes, nose,
mouth). Since demo data for faces is not cached yet, we show the API
pattern and construct example data to illustrate the response structure.

## Setup

In [ ]:
# !pip install oci-vision-ai[notebooks]

from oci_vision.core.client import VisionClient

client = VisionClient(demo=True)
print(f"Client ready (demo={client.is_demo})")

## Run the analysis

The `detect_faces()` method invokes the OCI Vision FACE_DETECTION feature.
In demo mode this returns `None` because no face detection demo data is
cached. Below we show the call and then build a synthetic example.

In [ ]:
# Demo mode -- returns None (no cached face data)
result = client.detect_faces("dog_closeup.jpg")
print(f"detect_faces() returned: {result}")
print()
print("In live mode, this returns a FaceDetectionResult with")
print("bounding boxes and facial landmark coordinates.")

## Explore the results

We construct a synthetic `FaceDetectionResult` to demonstrate the
data model and how landmarks work.

In [ ]:
from oci_vision.core.models import (
    FaceDetectionResult, DetectedFace, FaceLandmark,
    BoundingPolygon, Vertex
)

# Synthetic example -- two faces with landmarks
example_result = FaceDetectionResult(
    model_version="1.5.97",
    faces=[
        DetectedFace(
            confidence=0.9876,
            bounding_polygon=BoundingPolygon(
                normalized_vertices=[
                    Vertex(x=0.20, y=0.10),
                    Vertex(x=0.45, y=0.10),
                    Vertex(x=0.45, y=0.50),
                    Vertex(x=0.20, y=0.50),
                ]
            ),
            landmarks=[
                FaceLandmark(type="LEFT_EYE", x=0.28, y=0.25),
                FaceLandmark(type="RIGHT_EYE", x=0.38, y=0.25),
                FaceLandmark(type="NOSE_TIP", x=0.33, y=0.33),
                FaceLandmark(type="LEFT_MOUTH", x=0.29, y=0.40),
                FaceLandmark(type="RIGHT_MOUTH", x=0.37, y=0.40),
            ]
        ),
        DetectedFace(
            confidence=0.9234,
            bounding_polygon=BoundingPolygon(
                normalized_vertices=[
                    Vertex(x=0.55, y=0.15),
                    Vertex(x=0.80, y=0.15),
                    Vertex(x=0.80, y=0.55),
                    Vertex(x=0.55, y=0.55),
                ]
            ),
            landmarks=[
                FaceLandmark(type="LEFT_EYE", x=0.63, y=0.28),
                FaceLandmark(type="RIGHT_EYE", x=0.73, y=0.28),
                FaceLandmark(type="NOSE_TIP", x=0.68, y=0.37),
                FaceLandmark(type="LEFT_MOUTH", x=0.64, y=0.45),
                FaceLandmark(type="RIGHT_MOUTH", x=0.72, y=0.45),
            ]
        ),
    ]
)

print(f"Faces detected: {len(example_result.faces)}")
for i, face in enumerate(example_result.faces, 1):
    print(f"  Face {i}: confidence={face.confidence:.1%}, "
          f"landmarks={len(face.landmarks)}, "
          f"position={face.bounding_polygon.human_position(800, 600)}")

In [ ]:
# Detailed landmark listing
for i, face in enumerate(example_result.faces, 1):
    print(f"Face {i} landmarks:")
    for lm in face.landmarks:
        # Convert to pixel coordinates for an 800x600 image
        px_x = int(lm.x * 800)
        px_y = int(lm.y * 600)
        print(f"  {lm.type:15s} -> normalized ({lm.x:.2f}, {lm.y:.2f}) "
              f"-> pixels ({px_x}, {px_y})")
    print()

## Visualize

The renderer draws face bounding boxes with blue outlines and marks
each landmark as a filled circle.

In [ ]:
from oci_vision.core.models import AnalysisReport
from oci_vision.core.renderer import render_overlay
from PIL import Image

# Create a placeholder and render face overlays
placeholder = Image.new('RGB', (800, 600), color=(220, 220, 240))

report = AnalysisReport(
    image_path="synthetic_faces.jpg",
    faces=example_result
)

overlay = render_overlay(placeholder, report)
print("Blue boxes = face bounding boxes")
print("Blue dots  = facial landmarks (eyes, nose, mouth)")
overlay

## Under the hood

In [ ]:
import json

raw = example_result.model_dump()
print(json.dumps(raw, indent=2, default=str))

### Live API usage

```python
client = VisionClient()  # uses ~/.oci/config
result = client.detect_faces("photo_with_people.jpg")

for face in result.faces:
    print(f"Face at {face.bounding_polygon.human_position(w, h)} "
          f"(confidence: {face.confidence:.1%})")
    for lm in face.landmarks:
        print(f"  {lm.type}: ({lm.x:.3f}, {lm.y:.3f})")
```

## Try it yourself

1. **Live detection**: Switch to `VisionClient()` and pass a photo with
   people to `detect_faces()`.
2. **Landmark geometry**: Calculate inter-pupillary distance or face
   symmetry from landmark coordinates.
3. **Face cropping**: Use the bounding polygon with `to_pixels()` to
   crop individual faces from group photos.
4. **Multi-feature**: Combine face detection with classification using
   `client.analyze(img, features=['faces', 'classification'])`.